# DeFiPy: Python SDK for DeFi Analytics
## Chapter 7: Liquidity Trees and the Dawn of AMM Nets

### Listing 7.1: Uniswap V2 pool pricing

In [1]:
from defipy import ERC20, UniswapFactory, UniswapExchangeData, LPQuote, Join

tkn = ERC20("TKN", "0x111")
eth = ERC20("ETH", "0x09")
exchg_data = UniswapExchangeData(tkn0 = eth, tkn1 = tkn, symbol="LP", address="0x011")

factory = UniswapFactory("ETH pool factory", "0x2")
lp = factory.deploy(exchg_data)

Join().apply(lp, 'user', 1000, 100000)
lp.summary()

amt_eth_lp = LPQuote(False).get_lp_from_amount(lp, eth, 1)
print(f'1 {eth.token_name} token is worth {amt_eth_lp:.4f} LP tokens')

Exchange ETH-TKN (LP)
Reserves: ETH = 1000.0, TKN = 100000.0
Liquidity: 10000.0 

1 ETH token is worth 5.0088 LP tokens


### Listing 7.2: Uniswap V3 pool pricing

In [2]:
from defipy import UniV3Utils, UniV3Helper, UniswapFactory, UniswapExchangeData, Join, LPQuote, ERC20

eth = ERC20("ETH", "0x09")
tkn = ERC20("TKN", "0x111")

fee = UniV3Utils.FeeAmount.MEDIUM
tick_spacing = UniV3Utils.TICK_SPACINGS[fee]

exchg_data = UniswapExchangeData(tkn0 = eth, tkn1 = tkn, symbol="LP", 
                                   address="0x011", version = 'V3', 
                                   tick_spacing = tick_spacing, 
                                   fee = fee)

factory = UniswapFactory("ETH pool factory", "0x2")
lp = factory.deploy(exchg_data)

lwr_tick = UniV3Helper().get_price_tick(lp, -1, 100, 1000)
upr_tick = UniV3Helper().get_price_tick(lp, 1, 100, 1000)

Join().apply(lp, 'user', 1000, 100000, lwr_tick, upr_tick)
lp.summary()

amt_eth_lp = LPQuote(False).get_lp_from_amount(lp, eth, 1, lwr_tick, upr_tick)
print(f'1 {eth.token_name} token is worth {amt_eth_lp:.4f} LP tokens')

Exchange ETH-TKN (LP)
Real Reserves:   ETH = 992.1398580613097, TKN = 100000.00000000017
Gross Liquidity: 206257.82469781497 

1 ETH token is worth 103.6932 LP tokens


### Listing 7.3: Uniswap V2 liquidity tree

In [3]:
from defipy import *

dai1 = ERC20('DAI', "0x111")
tkn1 = ERC20('USDC', "0x09")
exchg_data = UniswapExchangeData(tkn0 = tkn1, tkn1 = dai1, symbol = "LP", address = "0x011")

TKN_amt = TokenDeltaModel(1000)

iVault1 = IndexVault('iVault1', "0x7")
factory = UniswapFactory("UniV2 pool factory", "0x2")
lp = factory.deploy(exchg_data)
Join().apply(lp, 'user', 100000, 100000)

tkn2 = ERC20('USDC', "0x09")
itkn1 = IndexERC20('iUSDC', "0x09", tkn1, lp)
exchg_data1 = UniswapExchangeData(tkn0 = tkn2, tkn1 = itkn1, symbol="LP1", address="0x012")
lp1 = factory.deploy(exchg_data1)
JoinTree().apply(lp1, 'user', iVault1, 10000)

# Re-balance LP price after JoinTree
SwapDeposit().apply(lp, dai1, 'user', lp.get_reserve(tkn1) - lp.get_reserve(dai1))

lp.summary()
lp1.summary()

Exchange USDC-DAI (LP)
Reserves: USDC = 110000.0, DAI = 110000.0
Liquidity: 109984.62065824166 

Exchange USDC-iUSDC (LP1)
Reserves: USDC = 9972.071706380653, iUSDC = 4873.5527472211
Liquidity: 6971.328242172881 



### Listing 7.4: Uniswap V2 liquidity tree test

In [4]:
# code block from 7.3

tkn_invest = 100
invested_user_nm = 'invested_user'

lp1_state = MarkovState(stochastic = True)

SwapIndexMint(iVault1, opposing = False).apply(lp, tkn1, invested_user_nm, tkn_invest)
mint_itkn1_deposit = lp1.convert_to_human(iVault1.index_tokens['iUSDC']['last_lp_deposit'])
lp1_state.next_state(mint_itkn1_deposit) 
SwapDeposit().apply(lp1, itkn1, invested_user_nm, mint_itkn1_deposit)

lp.summary()
lp1.summary()

lp_invest_track  = lp.get_liquidity_from_provider(invested_user_nm)
lp1_invest_track  = lp1.get_liquidity_from_provider(invested_user_nm)

# Redeem from parent
tkn_redeem_parent = LPQuote(False).get_amount_from_lp(lp, tkn1, lp_invest_track)

# Redeem from tree (child + parent)
itkn_redeem_child = LPQuote(False).get_amount_from_lp(lp1, itkn1, lp1_invest_track)
tkn_redeem_tree = LPQuote(False).get_amount_from_lp(lp, tkn1, itkn_redeem_child) 

print(f'{tkn_redeem_parent:.3f} USDC redeemed from {lp_invest_track:.3f} LP tokens (parent)')
print(f'{tkn_redeem_tree:.3f} USDC redeemed from {lp1_invest_track:.3f} LP1 tokens if (tree)')

Exchange USDC-DAI (LP)
Reserves: USDC = 110100.0, DAI = 110000.0
Liquidity: 110034.52722566498 

Exchange USDC-iUSDC (LP1)
Reserves: USDC = 9972.071706380653, iUSDC = 4923.4593146444195
Liquidity: 7006.878035281126 

99.700 USDC redeemed from 49.907 LP tokens (parent)
99.403 USDC redeemed from 35.550 LP1 tokens if (tree)


### Listing 7.5: Uniswap V2 liquidity tree simulation

In [5]:
# *************************
# *** Simulation
# *************************
n_sim_runs = 2000
seconds_year = 31536000
shape = 2000
scale = 0.0005

p_arr = np.random.gamma(shape = shape, scale = scale, size = n_sim_runs)

In [6]:
arb = CorrectReserves(lp, x0 = 1)
arb1 = Arbitrage(lp1, lp1_state) 

TKN_amt = TokenDeltaModel(1000)
TKN_amt_arb = TokenDeltaModel(100)

lp_direct_invest_arr = []; lp1_direct_invest_arr = []; lp1_tree_invest_arr = []; 
pTKN_DAI_arr = []; pTKN_iTKN_arr = []
fee_lp_arr  = []; fee_lp1_arr  = [];

for k in range(2000):

    #if(k % 100 == 0 and k != 0): print(f'Processing event {k}')
    
    # *****************************
    # ***** Parent Arbitrage ******
    # *****************************   
    arb.apply(p_arr[k])

    # *****************************
    # ***** Child Arbitrage ******
    # *****************************       
    amt_arb1 = TKN_amt_arb.delta()   
    arb1.apply(1, "user", amt_arb1)
    arb1.update_state(itkn1)    

    mint_tkn1_amt = 0.5*TKN_amt.delta()
    SwapIndexMint(iVault1, opposing = False).apply(lp, tkn1, "user", mint_tkn1_amt)
    mint_itkn1_deposit = lp1.convert_to_human(iVault1.index_tokens['iUSDC']['last_lp_deposit'])
    lp1_state.next_state(mint_itkn1_deposit)   
    vault_lp1_amt = lp1_state.get_current_state('dVault')  
    burned_itkn1_amt = lp1_state.get_current_state('dBurned') 

    ## WithdrawSwap burned token from parent LP
    if(burned_itkn1_amt > 0):
        total_tkn_w_swap = LPQuote(False).get_amount_from_lp(lp, tkn1, burned_itkn1_amt)
        amt_out = RemoveLiquidity().apply(lp, tkn1, "user", total_tkn_w_swap/2)    

    ## Balance LP1: TKN/iTKN
    if(vault_lp1_amt > 0):
        # A portion of aquired token is coming from newly minted, while the remainder is coming from held 
        amt_tkn = LPQuote(False).get_amount_from_lp(lp, tkn1, vault_lp1_amt) 
        price_tkn = amt_tkn/vault_lp1_amt 
        AddLiquidity(price_tkn).apply(lp1, itkn1, "user", vault_lp1_amt)        
    elif(vault_lp1_amt < 0):
        # A portion of removed token is getting held, while the remainder is getting burned
        RemoveLiquidity().apply(lp1, itkn1, "user", abs(vault_lp1_amt))  

    # *****************************
    # ***** Random Swapping ******
    # *****************************       
    Swap().apply(lp, tkn1, "user", TKN_amt.delta()) 
    Swap().apply(lp, dai1, "user", TKN_amt.delta()) 

    # conservatively assume 10% of tokens held outside vault are traded
    held_tokens = lp1_state.get_current_state('Held')
    if(held_tokens > 0):
        tradable_itkn1_amt = 0.1*held_tokens
        Swap().apply(lp1, tkn2, "user", LPQuote(False).get_amount_from_lp(lp, tkn1, tradable_itkn1_amt))  
        Swap().apply(lp1, itkn1, "user", tradable_itkn1_amt)

lp.summary()
lp1.summary()

Exchange USDC-DAI (LP)
Reserves: USDC = 169821.29189930757, DAI = 163392.95512566148
Liquidity: 158900.46354322633 

Exchange USDC-iUSDC (LP1)
Reserves: USDC = 28293.4802173205, iUSDC = 13782.923907956967
Liquidity: 16658.73808136816 

